# Notebook 13 — Build H₂ Pipeline Cost Functions

## Purpose

This notebook converts the standardized H₂ pipeline capacity–cost observations
created in Notebook 12 into continuous cost functions for later use in the
Geospatial-CANOE schema-construction workflow.

The notebook fits candidate regression functions to three pipeline cost
components:

- capital investment cost;
- fixed operating cost;
- variable operating cost.

Both linear and power functions are fitted to each component for diagnostic
comparison. The cost-function forms used by the model are then selected
explicitly:

| Cost component | Selected function |
|---|---|
| CAPEX | Power |
| Fixed OPEX | Linear |
| Variable OPEX | Linear |

The selected functions remain expressed on a **per-kilometre basis**. They are
not yet assigned to individual graph edges or multiplied by corridor distance.
That topology-aware mapping is deferred to Notebook 14.

---

## Input

The notebook reads:

`data_files/processed/costs/transport/pipelines/h2_pipeline/h2_pipeline_normalized_capacity_costs.csv`

This table contains standardized engineering observations for `H2_PIPE`,
including:

- pipeline diameter;
- annual H₂ transport capacity;
- CAPEX in CAD 2020 per kilometre;
- annual fixed OPEX in CAD 2020 per year per kilometre;
- annual variable OPEX in CAD 2020 per year per kilometre;
- source and cost-model metadata.

---

## Regression models

### Linear model

The linear function is

\[
C(Q) = mQ + b
\]

where:

- \(Q\) is annual pipeline capacity in tonnes of H₂ per year;
- \(m\) is the fitted slope;
- \(b\) is the fitted intercept;
- \(C(Q)\) is the relevant pipeline cost per kilometre.

### Power model

The power function is

\[
C(Q) = aQ^{\beta}
\]

where:

- \(a\) is the fitted coefficient;
- \(\beta\) is the fitted exponent.

The power model is estimated through a log–log transformation, but its
diagnostic metrics are calculated after transforming predictions back into the
original monetary units.

---

## Model diagnostics

Each candidate function is evaluated using:

- coefficient of determination, \(R^2\);
- root mean squared error;
- mean absolute error;
- maximum absolute error.

These diagnostics are retained for transparency, but model selection is
specified explicitly rather than determined automatically from a single
goodness-of-fit statistic.

---

## Output

The principal output is:

`data_files/processed/costs/transport/pipelines/h2_pipeline/h2_pipeline_cost_model_selection.csv`

The exported table contains one selected function for each cost component,
including:

- technology and commodity identifiers;
- cost type;
- selected model type;
- regression coefficients;
- valid observed capacity range;
- currency and source metadata;
- regression diagnostics.

This CSV is an intermediate, topology-independent cost definition.

Notebook 14 will combine these selected functions with canonical graph-edge
information and corridor distances to produce edge-specific pipeline cost
tables for schema construction.

---

## Workflow position

```text
Notebook 12
Standardize engineering capacity–cost observations
        ↓
Notebook 13
Fit and select continuous per-kilometre cost functions
        ↓
Notebook 14
Map cost functions to graph edges and multiply by distance
        ↓
build_schema.py
Encode CostInvest, CostFixed, CostVariable, and ETLSegment

In [1]:
# =============================================================================
# Imports and project discovery
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    """Locate the Geospatial-CANOE repository root.

    Searches upward from the current working directory for a parent directory
    containing the project ``data_files`` and ``scripts`` directories.

    Returns
    -------
    Path
        Absolute path to the repository root.

    Raises
    ------
    FileNotFoundError
        If the repository root cannot be located.
    """

    for candidate in [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]:
        if (
            (candidate / "data_files").is_dir()
            and (candidate / "scripts").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Geospatial-CANOE project root. "
        "Expected a parent directory containing both "
        "'data_files/' and 'scripts/'."
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace


In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

DATA_FILES = PROJECT_ROOT / "data_files"

PROCESSED_COSTS_DIR = (
    DATA_FILES
    / "processed"
    / "costs"
    / "transport"
    / "pipelines"
    / "h2_pipeline"
)

INPUT_CAPACITY_COST_PATH = (
    PROCESSED_COSTS_DIR
    / "h2_pipeline_normalized_capacity_costs.csv"
)

OUTPUT_REGRESSION_PATH = (
    PROCESSED_COSTS_DIR
    / "h2_pipeline_cost_regressions.csv"
)

OUTPUT_PREDICTION_PATH = (
    PROCESSED_COSTS_DIR
    / "h2_pipeline_cost_predictions.csv"
)

OUTPUT_MODEL_SELECTION_PATH = (
    PROCESSED_COSTS_DIR
    / "h2_pipeline_cost_model_selection.csv"
)

print("Cost-model paths:")
print(f"  Input capacity costs: {INPUT_CAPACITY_COST_PATH}")
print(f"  Regression output:    {OUTPUT_REGRESSION_PATH}")
print(f"  Prediction output:    {OUTPUT_PREDICTION_PATH}")
print(f"  Selection output:     {OUTPUT_MODEL_SELECTION_PATH}")

Cost-model paths:
  Input capacity costs: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_normalized_capacity_costs.csv
  Regression output:    C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_cost_regressions.csv
  Prediction output:    C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_cost_predictions.csv
  Selection output:     C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_cost_model_selection.csv


In [3]:
# =============================================================================
# Validate input path and prepare output directories
# =============================================================================

if not INPUT_CAPACITY_COST_PATH.exists():
    raise FileNotFoundError(
        "Processed H2 pipeline capacity-cost table not found: "
        f"{INPUT_CAPACITY_COST_PATH}"
    )

PROCESSED_COSTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Cost-model paths validated:")
print(f"  Input file exists:   {INPUT_CAPACITY_COST_PATH.name}")
print(f"  Output folder ready: {PROCESSED_COSTS_DIR}")

Cost-model paths validated:
  Input file exists:   h2_pipeline_normalized_capacity_costs.csv
  Output folder ready: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline


In [4]:
# =============================================================================
# Load processed H2 pipeline capacity-cost table
# =============================================================================

def load_capacity_cost_table(
    input_path: Path,
) -> pd.DataFrame:
    """Load the processed H2 pipeline capacity-cost table.

    Parameters
    ----------
    input_path : Path
        Path to the processed capacity-cost CSV.

    Returns
    -------
    pd.DataFrame
        Loaded pipeline capacity-cost table.

    Raises
    ------
    ValueError
        If the loaded table is empty.
    """

    cost_table = pd.read_csv(
        input_path,
        encoding="utf-8",
    )

    if cost_table.empty:
        raise ValueError(
            "Processed H2 pipeline capacity-cost table is empty."
        )

    return cost_table


pipeline_capacity_costs = load_capacity_cost_table(
    input_path=INPUT_CAPACITY_COST_PATH,
)

print(
    "Loaded H2 pipeline capacity-cost table: "
    f"{pipeline_capacity_costs.shape[0]:,} rows × "
    f"{pipeline_capacity_costs.shape[1]:,} columns"
)

print("\nColumns:")
for column in pipeline_capacity_costs.columns:
    print(f"  - {column}")

display(pipeline_capacity_costs)

Loaded H2 pipeline capacity-cost table: 13 rows × 11 columns

Columns:
  - technology
  - commodity
  - diameter_in
  - capacity_t_h2_per_year
  - total_capex_cad2020_per_km
  - total_variable_opex_cad2020_per_year_per_km
  - total_fixed_opex_cad2020_per_year_per_km
  - currency
  - currency_year
  - source_workbook
  - cost_model_version


,technology,commodity,diameter_in,capacity_t_h2_per_year,total_capex_cad2020_per_km,total_variable_opex_cad2020_per_year_per_km,total_fixed_opex_cad2020_per_year_per_km,currency,currency_year,source_workbook,cost_model_version
0,H2_PIPE,h2,8,5.397953e+04,1.394121e+06,10333.906985,40138.383237,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
1,H2_PIPE,h2,10,9.429835e+04,1.632461e+06,17901.573988,60354.158581,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
2,H2_PIPE,h2,12,1.487501e+05,1.880928e+06,27961.113404,80401.712562,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
3,H2_PIPE,h2,14,2.186876e+05,2.140113e+06,40711.907844,100426.340615,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
4,H2_PIPE,h2,16,3.053543e+05,2.410694e+06,56335.036261,120535.913145,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
5,H2_PIPE,h2,18,4.099070e+05,2.694416e+06,75167.744427,140829.069306,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
6,H2_PIPE,h2,20,5.334320e+05,2.991624e+06,97417.863378,161376.679926,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
7,H2_PIPE,h2,22,6.769566e+05,3.303302e+06,123270.426166,182237.909868,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
8,H2_PIPE,h2,24,8.414575e+05,3.630651e+06,152901.400879,203465.076877,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1
9,H2_PIPE,h2,26,1.027868e+06,3.975046e+06,186478.906891,225106.259138,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1


In [5]:
# =============================================================================
# Define and validate pipeline cost-model inputs
# =============================================================================

CAPACITY_COLUMN = "capacity_t_h2_per_year"

COST_COLUMNS = {
    "capex": "total_capex_cad2020_per_km",
    "variable_opex": (
        "total_variable_opex_cad2020_per_year_per_km"
    ),
    "fixed_opex": (
        "total_fixed_opex_cad2020_per_year_per_km"
    ),
}

METADATA_COLUMNS = [
    "technology",
    "commodity",
    "currency",
    "currency_year",
    "source_workbook",
    "cost_model_version",
]

REQUIRED_COLUMNS = [
    *METADATA_COLUMNS,
    "diameter_in",
    CAPACITY_COLUMN,
    *COST_COLUMNS.values(),
]


def validate_capacity_cost_table(
    cost_table: pd.DataFrame,
    required_columns: list[str],
    metadata_columns: list[str],
    capacity_column: str,
    cost_columns: dict[str, str],
) -> None:
    """Validate the processed pipeline capacity-cost input table."""

    missing_columns = [
        column
        for column in required_columns
        if column not in cost_table.columns
    ]

    if missing_columns:
        raise ValueError(
            "Pipeline capacity-cost table is missing required columns: "
            f"{missing_columns}"
        )

    duplicated_columns = (
        cost_table.columns[
            cost_table.columns.duplicated()
        ]
        .tolist()
    )

    if duplicated_columns:
        raise ValueError(
            "Pipeline capacity-cost table contains duplicate columns: "
            f"{duplicated_columns}"
        )

    numeric_columns = [
        "diameter_in",
        capacity_column,
        *cost_columns.values(),
    ]

    for column in numeric_columns:
        numeric_values = pd.to_numeric(
            cost_table[column],
            errors="coerce",
        )

        if numeric_values.isna().any():
            raise ValueError(
                f"Column '{column}' contains missing or non-numeric values."
            )

        if (numeric_values <= 0).any():
            raise ValueError(
                f"Column '{column}' must contain only positive values."
            )

    duplicated_cases = cost_table.duplicated(
        subset=[
            "diameter_in",
            capacity_column,
        ],
        keep=False,
    )

    if duplicated_cases.any():
        raise ValueError(
            "Duplicate diameter-capacity cases were found:\n"
            f"{cost_table.loc[duplicated_cases, ['diameter_in', capacity_column]]}"
        )

    for column in metadata_columns:
        unique_values = cost_table[column].dropna().unique()

        if len(unique_values) != 1:
            raise ValueError(
                f"Metadata column '{column}' must contain exactly one "
                f"value; found {unique_values.tolist()}."
            )


validate_capacity_cost_table(
    cost_table=pipeline_capacity_costs,
    required_columns=REQUIRED_COLUMNS,
    metadata_columns=METADATA_COLUMNS,
    capacity_column=CAPACITY_COLUMN,
    cost_columns=COST_COLUMNS,
)

print("Pipeline capacity-cost table validated.")
print(f"  Technology: {pipeline_capacity_costs['technology'].iloc[0]}")
print(f"  Commodity:  {pipeline_capacity_costs['commodity'].iloc[0]}")
print(
    "  Capacity range: "
    f"{pipeline_capacity_costs[CAPACITY_COLUMN].min():,.0f} to "
    f"{pipeline_capacity_costs[CAPACITY_COLUMN].max():,.0f} "
    "t H2/year"
)

Pipeline capacity-cost table validated.
  Technology: H2_PIPE
  Commodity:  h2
  Capacity range: 53,980 to 1,727,345 t H2/year


In [6]:
# =============================================================================
# Regression diagnostic metrics
# =============================================================================

def calculate_regression_metrics(
    observed: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    """Calculate regression diagnostics in the original cost units.

    Parameters
    ----------
    observed : np.ndarray
        Observed cost values.
    predicted : np.ndarray
        Cost values predicted by the fitted model.

    Returns
    -------
    dict[str, float]
        Coefficient of determination, root mean squared error,
        mean absolute error, and maximum absolute error.

    Raises
    ------
    ValueError
        If observed and predicted arrays differ in length, are empty,
        or contain non-finite values.
    """

    observed = np.asarray(
        observed,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    if observed.shape != predicted.shape:
        raise ValueError(
            "Observed and predicted arrays must have the same shape."
        )

    if observed.size == 0:
        raise ValueError(
            "Regression metrics cannot be calculated from empty arrays."
        )

    if not np.isfinite(observed).all():
        raise ValueError(
            "Observed values contain non-finite values."
        )

    if not np.isfinite(predicted).all():
        raise ValueError(
            "Predicted values contain non-finite values."
        )

    residuals = observed - predicted

    residual_sum_of_squares = float(
        np.sum(residuals ** 2)
    )

    total_sum_of_squares = float(
        np.sum(
            (observed - observed.mean()) ** 2
        )
    )

    r_squared = (
        np.nan
        if total_sum_of_squares == 0
        else 1.0
        - residual_sum_of_squares
        / total_sum_of_squares
    )

    rmse = float(
        np.sqrt(
            np.mean(residuals ** 2)
        )
    )

    mae = float(
        np.mean(
            np.abs(residuals)
        )
    )

    maximum_absolute_error = float(
        np.max(
            np.abs(residuals)
        )
    )

    return {
        "r_squared": r_squared,
        "rmse": rmse,
        "mae": mae,
        "maximum_absolute_error": maximum_absolute_error,
    }

In [7]:
# =============================================================================
# Fit linear cost model
# =============================================================================

def fit_linear_cost_model(
    capacity: np.ndarray,
    cost: np.ndarray,
) -> dict[str, float | str]:
    """Fit a linear cost function of the form y = slope*x + intercept.

    Parameters
    ----------
    capacity : np.ndarray
        Pipeline capacity observations.
    cost : np.ndarray
        Corresponding cost observations.

    Returns
    -------
    dict[str, float | str]
        Linear-model coefficients, equation form, and diagnostic metrics.

    Raises
    ------
    ValueError
        If the arrays differ in shape, contain fewer than two observations,
        or contain non-finite values.
    """

    capacity = np.asarray(
        capacity,
        dtype=float,
    )

    cost = np.asarray(
        cost,
        dtype=float,
    )

    if capacity.shape != cost.shape:
        raise ValueError(
            "Capacity and cost arrays must have the same shape."
        )

    if capacity.size < 2:
        raise ValueError(
            "At least two observations are required to fit a linear model."
        )

    if not np.isfinite(capacity).all():
        raise ValueError(
            "Capacity values contain non-finite values."
        )

    if not np.isfinite(cost).all():
        raise ValueError(
            "Cost values contain non-finite values."
        )

    if np.unique(capacity).size < 2:
        raise ValueError(
            "Linear regression requires at least two unique capacities."
        )

    slope, intercept = np.polyfit(
        capacity,
        cost,
        deg=1,
    )

    predicted = (
        slope * capacity
        + intercept
    )

    metrics = calculate_regression_metrics(
        observed=cost,
        predicted=predicted,
    )

    return {
        "model_type": "linear",
        "equation": "y = slope * x + intercept",
        "slope": float(slope),
        "intercept": float(intercept),
        "coefficient": np.nan,
        "exponent": np.nan,
        **metrics,
    }

In [8]:
# =============================================================================
# Fit power cost model
# =============================================================================

def fit_power_cost_model(
    capacity: np.ndarray,
    cost: np.ndarray,
) -> dict[str, float | str]:
    """Fit a power cost function of the form y = coefficient*x**exponent.

    The model is estimated by fitting a linear relationship between the
    natural logarithms of capacity and cost. Regression diagnostics are then
    calculated using predicted values transformed back to the original cost
    units.

    Parameters
    ----------
    capacity : np.ndarray
        Pipeline capacity observations.
    cost : np.ndarray
        Corresponding cost observations.

    Returns
    -------
    dict[str, float | str]
        Power-model coefficients, equation form, and diagnostic metrics.

    Raises
    ------
    ValueError
        If the arrays differ in shape, contain fewer than two observations,
        contain non-finite values, or include non-positive values.
    """

    capacity = np.asarray(
        capacity,
        dtype=float,
    )

    cost = np.asarray(
        cost,
        dtype=float,
    )

    if capacity.shape != cost.shape:
        raise ValueError(
            "Capacity and cost arrays must have the same shape."
        )

    if capacity.size < 2:
        raise ValueError(
            "At least two observations are required to fit a power model."
        )

    if not np.isfinite(capacity).all():
        raise ValueError(
            "Capacity values contain non-finite values."
        )

    if not np.isfinite(cost).all():
        raise ValueError(
            "Cost values contain non-finite values."
        )

    if np.any(capacity <= 0):
        raise ValueError(
            "Power regression requires strictly positive capacity values."
        )

    if np.any(cost <= 0):
        raise ValueError(
            "Power regression requires strictly positive cost values."
        )

    if np.unique(capacity).size < 2:
        raise ValueError(
            "Power regression requires at least two unique capacities."
        )

    log_capacity = np.log(capacity)
    log_cost = np.log(cost)

    exponent, log_coefficient = np.polyfit(
        log_capacity,
        log_cost,
        deg=1,
    )

    coefficient = float(
        np.exp(log_coefficient)
    )

    predicted = (
        coefficient
        * capacity ** exponent
    )

    metrics = calculate_regression_metrics(
        observed=cost,
        predicted=predicted,
    )

    return {
        "model_type": "power",
        "equation": "y = coefficient * x ** exponent",
        "slope": np.nan,
        "intercept": np.nan,
        "coefficient": coefficient,
        "exponent": float(exponent),
        **metrics,
    }

In [9]:
# =============================================================================
# Fit candidate models to each pipeline cost dataset
# =============================================================================

def fit_pipeline_cost_models(
    cost_table: pd.DataFrame,
    capacity_column: str,
    cost_columns: dict[str, str],
) -> pd.DataFrame:
    """Fit linear and power functions to each pipeline cost dataset.

    Parameters
    ----------
    cost_table : pd.DataFrame
        Validated pipeline capacity-cost observations.
    capacity_column : str
        Column containing annual pipeline capacity.
    cost_columns : dict[str, str]
        Mapping from cost type to observed cost column.

    Returns
    -------
    pd.DataFrame
        Long-form regression summary with one row for each combination of
        cost type and candidate model type.
    """

    capacity = (
        pd.to_numeric(
            cost_table[capacity_column],
            errors="raise",
        )
        .to_numpy(dtype=float)
    )

    metadata = {
        column: cost_table[column].iloc[0]
        for column in METADATA_COLUMNS
    }

    regression_rows = []

    for cost_type, cost_column in cost_columns.items():
        cost = (
            pd.to_numeric(
                cost_table[cost_column],
                errors="raise",
            )
            .to_numpy(dtype=float)
        )

        fitted_models = [
            fit_linear_cost_model(
                capacity=capacity,
                cost=cost,
            ),
            fit_power_cost_model(
                capacity=capacity,
                cost=cost,
            ),
        ]

        for fitted_model in fitted_models:
            regression_rows.append(
                {
                    **metadata,
                    "cost_type": cost_type,
                    "capacity_column": capacity_column,
                    "cost_column": cost_column,
                    "model_type": fitted_model["model_type"],
                    "equation": fitted_model["equation"],
                    "slope": fitted_model["slope"],
                    "intercept": fitted_model["intercept"],
                    "coefficient": fitted_model["coefficient"],
                    "exponent": fitted_model["exponent"],
                    "capacity_min": float(capacity.min()),
                    "capacity_max": float(capacity.max()),
                    "n_observations": int(capacity.size),
                    "r_squared": fitted_model["r_squared"],
                    "rmse": fitted_model["rmse"],
                    "mae": fitted_model["mae"],
                    "maximum_absolute_error": (
                        fitted_model["maximum_absolute_error"]
                    ),
                }
            )

    regression_table = (
        pd.DataFrame(regression_rows)
        .sort_values(
            [
                "cost_type",
                "model_type",
            ]
        )
        .reset_index(drop=True)
    )

    return regression_table


pipeline_cost_regressions = fit_pipeline_cost_models(
    cost_table=pipeline_capacity_costs,
    capacity_column=CAPACITY_COLUMN,
    cost_columns=COST_COLUMNS,
)

print(
    "Fitted pipeline cost models: "
    f"{len(pipeline_cost_regressions):,}"
)

display(
    pipeline_cost_regressions[
        [
            "cost_type",
            "model_type",
            "slope",
            "intercept",
            "coefficient",
            "exponent",
            "r_squared",
            "rmse",
            "mae",
            "maximum_absolute_error",
        ]
    ]
)

Fitted pipeline cost models: 6


,cost_type,model_type,slope,intercept,coefficient,exponent,r_squared,rmse,mae,maximum_absolute_error
0,capex,linear,2.164644,1.639262e+06,NaN,NaN,0.976199,178911.865250,154432.997089,361987.107207
1,capex,power,NaN,NaN,20885.563742,0.379112,0.989806,117088.833518,93166.572740,291189.508497
2,fixed_opex,linear,0.144804,6.604113e+04,NaN,NaN,0.955663,16509.699722,14234.648449,33719.210908
3,fixed_opex,power,NaN,NaN,102.160737,0.556838,0.996125,4881.018792,4007.585101,10766.355578
4,variable_opex,linear,0.180341,1.087392e+03,NaN,NaN,0.999996,180.182197,137.447222,488.181829
5,variable_opex,power,NaN,NaN,0.230526,0.982698,0.999953,656.517845,417.273322,1885.909579


In [10]:
# =============================================================================
# Select cost-function forms for schema construction
# =============================================================================

SELECTED_MODEL_TYPES = {
    "capex": "power",
    "variable_opex": "linear",
    "fixed_opex": "linear",
}


def select_pipeline_cost_models(
    regression_table: pd.DataFrame,
    selected_model_types: dict[str, str],
) -> pd.DataFrame:
    """Select one fitted function for each pipeline cost component.

    Parameters
    ----------
    regression_table : pd.DataFrame
        Candidate linear and power regressions for each cost type.
    selected_model_types : dict[str, str]
        Mapping from cost type to the model form selected for schema
        construction.

    Returns
    -------
    pd.DataFrame
        Regression table containing one selected model per cost type.

    Raises
    ------
    ValueError
        If a requested cost type or model type is unavailable, or if the
        selection does not identify exactly one row.
    """

    selected_rows = []

    for cost_type, model_type in selected_model_types.items():
        matching_rows = regression_table.loc[
            (
                regression_table["cost_type"]
                == cost_type
            )
            & (
                regression_table["model_type"]
                == model_type
            )
        ].copy()

        if len(matching_rows) != 1:
            raise ValueError(
                "Expected exactly one fitted model for "
                f"cost_type='{cost_type}' and "
                f"model_type='{model_type}', but found "
                f"{len(matching_rows)}."
            )

        selected_rows.append(matching_rows)

    selected_models = (
        pd.concat(
            selected_rows,
            ignore_index=True,
        )
        .sort_values("cost_type")
        .reset_index(drop=True)
    )

    selected_models.insert(
        selected_models.columns.get_loc("model_type") + 1,
        "is_selected",
        True,
    )

    return selected_models


selected_pipeline_cost_models = select_pipeline_cost_models(
    regression_table=pipeline_cost_regressions,
    selected_model_types=SELECTED_MODEL_TYPES,
)

print("Selected pipeline cost models:")

display(
    selected_pipeline_cost_models[
        [
            "cost_type",
            "model_type",
            "slope",
            "intercept",
            "coefficient",
            "exponent",
            "r_squared",
            "rmse",
            "mae",
        ]
    ]
)

Selected pipeline cost models:


,cost_type,model_type,slope,intercept,coefficient,exponent,r_squared,rmse,mae
0,capex,power,NaN,NaN,20885.563742,0.379112,0.989806,117088.833518,93166.572740
1,fixed_opex,linear,0.144804,66041.133024,NaN,NaN,0.955663,16509.699722,14234.648449
2,variable_opex,linear,0.180341,1087.392415,NaN,NaN,0.999996,180.182197,137.447222


In [11]:
# =============================================================================
# Export selected pipeline cost models
# =============================================================================

def export_selected_cost_models(
    selected_models: pd.DataFrame,
    output_path: Path,
) -> Path:
    """Export the selected pipeline cost functions to CSV.

    Parameters
    ----------
    selected_models : pd.DataFrame
        Selected regression rows containing one model for each cost type.
    output_path : Path
        Destination CSV path.

    Returns
    -------
    Path
        Path to the exported CSV.

    Raises
    ------
    ValueError
        If the selected-model table is empty, contains duplicate cost types,
        or does not contain the required cost components.
    OSError
        If the CSV is not created successfully.
    """

    required_cost_types = {
        "capex",
        "variable_opex",
        "fixed_opex",
    }

    if selected_models.empty:
        raise ValueError(
            "Selected pipeline cost-model table is empty."
        )

    selected_cost_types = set(
        selected_models["cost_type"].tolist()
    )

    missing_cost_types = (
        required_cost_types
        - selected_cost_types
    )

    unexpected_cost_types = (
        selected_cost_types
        - required_cost_types
    )

    if missing_cost_types:
        raise ValueError(
            "Selected pipeline cost-model table is missing cost types: "
            f"{sorted(missing_cost_types)}"
        )

    if unexpected_cost_types:
        raise ValueError(
            "Selected pipeline cost-model table contains unexpected "
            f"cost types: {sorted(unexpected_cost_types)}"
        )

    duplicated_cost_types = (
        selected_models["cost_type"]
        .duplicated(keep=False)
    )

    if duplicated_cost_types.any():
        duplicates = (
            selected_models.loc[
                duplicated_cost_types,
                "cost_type",
            ]
            .tolist()
        )

        raise ValueError(
            "Selected pipeline cost-model table contains duplicate "
            f"cost types: {duplicates}"
        )

    export_columns = [
        "technology",
        "commodity",
        "cost_type",
        "model_type",
        "equation",
        "slope",
        "intercept",
        "coefficient",
        "exponent",
        "capacity_min",
        "capacity_max",
        "capacity_column",
        "cost_column",
        "currency",
        "currency_year",
        "source_workbook",
        "cost_model_version",
        "r_squared",
        "rmse",
        "mae",
        "maximum_absolute_error",
    ]

    missing_export_columns = [
        column
        for column in export_columns
        if column not in selected_models.columns
    ]

    if missing_export_columns:
        raise ValueError(
            "Selected pipeline cost-model table is missing export "
            f"columns: {missing_export_columns}"
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    export_table = (
        selected_models[export_columns]
        .sort_values("cost_type")
        .reset_index(drop=True)
    )

    export_table.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    if not output_path.exists():
        raise OSError(
            "Selected pipeline cost-model CSV was not created: "
            f"{output_path}"
        )

    return output_path


selected_cost_model_path = export_selected_cost_models(
    selected_models=selected_pipeline_cost_models,
    output_path=OUTPUT_MODEL_SELECTION_PATH,
)

print("Exported selected pipeline cost models:")
print(f"  Path: {selected_cost_model_path}")
print(f"  Rows: {len(selected_pipeline_cost_models):,}")

display(
    pd.read_csv(
        selected_cost_model_path,
        encoding="utf-8",
    )
)

Exported selected pipeline cost models:
  Path: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_cost_model_selection.csv
  Rows: 3


,technology,commodity,cost_type,model_type,equation,slope,intercept,coefficient,exponent,capacity_min,...,capacity_column,cost_column,currency,currency_year,source_workbook,cost_model_version,r_squared,rmse,mae,maximum_absolute_error
0,H2_PIPE,h2,capex,power,y = coefficient * x ** exponent,NaN,NaN,20885.563742,0.379112,53979.525993,...,capacity_t_h2_per_year,total_capex_cad2020_per_km,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1,0.989806,117088.833518,93166.572740,291189.508497
1,H2_PIPE,h2,fixed_opex,linear,y = slope * x + intercept,0.144804,66041.133024,NaN,NaN,53979.525993,...,capacity_t_h2_per_year,total_fixed_opex_cad2020_per_year_per_km,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1,0.955663,16509.699722,14234.648449,33719.210908
2,H2_PIPE,h2,variable_opex,linear,y = slope * x + intercept,0.180341,1087.392415,NaN,NaN,53979.525993,...,capacity_t_h2_per_year,total_variable_opex_cad2020_per_year_per_km,CAD,2020,h2_pipeline_costs_master_v2.xlsx,v1,0.999996,180.182197,137.447222,488.181829
